# CCA Lab: Being Clear and Direct
**Course:** Building with the Claude API — Anthropic Skilljar  
**Lesson:** Being Clear and Direct  
**Lab version:** 1.0 — Revised to 98%+ CCA coverage standard  

---

## What this lab covers

This lab operationalizes the lesson's core teaching: **prompt quality is an architectural decision, not a stylistic preference.**  
You will build, measure, and iterate a prompt through three versions — vague → clear → constrained — and collect objective evidence that each version is better than the last.

**CCA domains exercised:**
- Prompt Engineering & Optimization (primary)
- Evaluation & Quality Assurance (measurement cells)
- Production Architecture & Reliability (system prompt cell, failure mode cell)
- Cost, Latency & Performance Optimization (token and length analysis)

**By the end of this lab you will be able to:**
1. Classify a prompt as vague, clear, or constrained using boolean properties
2. Demonstrate that improving prompt clarity produces measurably better outputs
3. Use the `system` parameter correctly and explain when it is preferred over the user turn
4. Identify and document the production failure modes of vague prompts
5. Apply the write → evaluate → improve → re-evaluate cycle as a repeatable process

---
## Section 1: Prerequisites and environment setup
## CCA objective demonstrated: Environment validation and API key management

In [ ]:
# Prerequisites check
# Run this cell first. Green checkmarks = ready to proceed. Red X = fix before continuing.

import sys
import os
import importlib

checks = {
    'Python >= 3.8': sys.version_info >= (3, 8),
    'anthropic package installed': importlib.util.find_spec('anthropic') is not None,
    'ANTHROPIC_API_KEY set': bool(os.environ.get('ANTHROPIC_API_KEY', '')),
}

all_pass = True
for check, result in checks.items():
    icon = '✅' if result else '❌'
    print(f"{icon}  {check}")
    if not result:
        all_pass = False

if not all_pass:
    print()
    print('== Fix instructions ==')
    print('Install anthropic:  pip install anthropic')
    print()
    print('Set API key:')
    print('  macOS/Linux:  export ANTHROPIC_API_KEY="sk-ant-..."')
    print('  Windows CMD:  set ANTHROPIC_API_KEY=sk-ant-...')
    print('  Windows PS:   $env:ANTHROPIC_API_KEY="sk-ant-..."')
    print()
    print('After setting the key, restart the kernel and re-run this cell.')
else:
    print()
    print('All checks passed — ready to proceed.')

---
## Section 2: API glossary and reference
## CCA objective demonstrated: Fluency with Claude API parameters and their architectural roles

Before writing any prompts, read this reference. Every parameter listed here will be used in this lab.

| Parameter | Type | Required | Description | CCA note |
|---|---|---|---|---|
| `model` | string | Yes | Which Claude model to use. e.g. `claude-sonnet-4-6` | Model selection is an architectural decision — affects cost, latency, and capability ceiling |
| `max_tokens` | int | Yes | Maximum tokens in the response. Hard ceiling, not a target. | Omitting this causes an API error. See Cell 6 for the intentional demo. |
| `messages` | list | Yes | List of `{role, content}` dicts. Role is `user` or `assistant`. | The conversation history. In single-turn calls, always one user message. |
| `system` | string | No | Instructions that scope Claude's behavior for the entire session. | Preferred over user-turn instructions for role, format, and persistent constraints. See Section 5. |
| `temperature` | float | No | 0.0–1.0. Controls output randomness. Default ~1.0. | Lower = more deterministic. Set to 0.0 for classification and structured tasks. |
| `response.content` | list | — | List of content blocks in the response. Usually one `TextBlock`. | Always a list — even for single responses. Access text via `response.content[0].text`. |
| `response.usage` | object | — | Token counts: `input_tokens`, `output_tokens`. | Use to measure and optimize token budget. |
| `response.stop_reason` | string | — | Why generation stopped: `end_turn`, `max_tokens`, `stop_sequence`. | `max_tokens` here means the response was cut off — increase limit or shorten prompt. |

---
## Section 3: Client setup and the `ask()` helper
## CCA objective demonstrated: Encapsulating API calls in a reusable helper with logging

In [ ]:
import anthropic
import json
import time

# Initialize the Anthropic client.
# The client reads ANTHROPIC_API_KEY from the environment automatically.
# You never hard-code an API key in source code.
client = anthropic.Anthropic()

# Usage log — accumulates token counts across all calls in this session.
# CCA principle: always instrument your API usage. You cannot optimize what you do not measure.
usage_log = []

def ask(
    prompt: str,
    system: str = None,
    model: str = "claude-sonnet-4-6",
    max_tokens: int = 1024,
    temperature: float = 1.0,
    label: str = ""
) -> str:
    """
    Send a single-turn message to Claude and return the response text.

    Statement-by-statement explanation:

    Parameters
    ----------
    prompt      : The user-turn instruction or question.
    system      : Optional system prompt — sets Claude's role and persistent constraints.
                  If None, no system prompt is sent and Claude uses its default behavior.
    model       : The Claude model identifier string.
    max_tokens  : Hard ceiling on response length. Required by the API.
    temperature : Randomness dial. 0.0 = deterministic, 1.0 = default variation.
    label       : A human-readable tag logged with usage data for analysis.
    """

    # Build keyword arguments. system is optional — only include it if provided.
    kwargs = dict(
        model=model,
        max_tokens=max_tokens,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}],
    )
    # The system parameter sits outside the messages list.
    # It is a top-level API parameter, not a message role.
    if system:
        kwargs["system"] = system

    # Make the API call and capture the full response object.
    response = client.messages.create(**kwargs)

    # response.content is always a list of content blocks.
    # For text responses there is exactly one block: a TextBlock.
    # We access it at index [0] and read its .text attribute.
    text = response.content[0].text

    # Log token usage for later analysis.
    # response.usage contains input_tokens and output_tokens.
    usage_log.append({
        "label": label,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "stop_reason": response.stop_reason,
        "total_tokens": response.usage.input_tokens + response.usage.output_tokens,
    })

    return text

print("Client initialized. ask() helper ready.")
print(f"Default model: claude-sonnet-4-6")

---
## Section 4: Intentional error demonstration — what happens when `max_tokens` is omitted
## CCA objective demonstrated: Understanding required parameters and reading API error messages

In [ ]:
# This cell deliberately triggers a BadRequestError by omitting max_tokens.
# Read the error message carefully — it is the API telling you exactly what is wrong.
# CCA principle: required parameters must be present. The API will not guess defaults for them.

print("Attempting API call without max_tokens...")
print()

try:
    # Intentionally calling client.messages.create() without max_tokens.
    # This will raise an anthropic.BadRequestError.
    broken_response = client.messages.create(
        model="claude-sonnet-4-6",
        messages=[{"role": "user", "content": "Hello"}]
        # max_tokens is missing — this is the deliberate error
    )
except Exception as e:
    print(f"Error type : {type(e).__name__}")
    print(f"Error message: {e}")
    print()
    print("Lesson: max_tokens is required. Always include it explicitly.")
    print("Production tip: wrap API calls in try/except and log error type + message.")

---
## Section 5: The `system` parameter — what it is, when to use it, and why it matters
## CCA objective demonstrated: Correct use of the system parameter vs. user-turn instructions

### When to use `system` vs. the user message

The **system parameter** sets Claude's persistent context for the entire session — role, output format, constraints, audience, and tone. It is processed before the user message and frames how Claude interprets everything that follows.

The **user message** contains the specific task, question, or content for this particular call.

| Instruction type | Where it belongs | Why |
|---|---|---|
| "You are a technical instructor for mid-level engineers" | `system` | Role is persistent — applies to every response |
| "Always respond in JSON with keys: answer, confidence" | `system` | Format constraint is persistent |
| "Never include legal advice" | `system` | Negative constraint must hold for every response |
| "Explain recursion" | `messages` user turn | This is the specific task for this call |
| "Here is the document to summarize: [text]" | `messages` user turn | Input content belongs in the user turn |

**CCA architectural principle:** System prompts are templates, not fixed strings. In multi-tenant or multi-role applications, inject role-specific context dynamically into the system prompt at request time — not into the user message.

In [ ]:
# Demonstration: same user prompt, two different system contexts, different outputs.
# This shows that the system parameter controls the interpretive frame Claude uses.

USER_PROMPT = "What is an API?"

# System context A: technical expert audience
SYSTEM_A = "You are a senior software architect writing documentation for experienced engineers. Use precise technical terminology. Be concise. Maximum 3 sentences."

# System context B: beginner audience
SYSTEM_B = "You are a patient instructor teaching complete beginners with no coding background. Use everyday analogies. Avoid jargon. Maximum 3 sentences."

print("=" * 60)
print("USER PROMPT (identical in both calls):")
print(f'  "{USER_PROMPT}"')
print()

response_a = ask(USER_PROMPT, system=SYSTEM_A, max_tokens=200, label="system_demo_expert")
print("SYSTEM A — Expert audience:")
print(response_a)
print()

response_b = ask(USER_PROMPT, system=SYSTEM_B, max_tokens=200, label="system_demo_beginner")
print("SYSTEM B — Beginner audience:")
print(response_b)
print()
print("Observation: identical user prompt, identical model, different system = different output.")
print("The system parameter controls the interpretive frame — not just the tone.")

---
## Section 5b: Multi-turn conversation — how clarity propagates across message history
## CCA objective demonstrated: Understanding the `messages` list as an accumulating conversation history

### How the `messages` list works across turns

Every API call in this lab so far has sent a single message:
```python
messages=[{"role": "user", "content": prompt}]
```

This is a **single-turn call** — Claude has no memory of any prior exchange.  
For multi-turn conversations, you accumulate the full history in the list and send it with every call:

```python
messages = [
    {"role": "user",      "content": "Tell me about APIs."},         # Turn 1 — vague
    {"role": "assistant", "content": "<Claude's first response>"},   # Turn 1 response
    {"role": "user",      "content": "Give me a Python code example with the requests library, under 50 words."},  # Turn 2 — corrective constraint
]
```

**Why this matters for clarity:**  
A vague Turn 1 produces a vague response. Turn 2 can correct that — but Claude only knows what you send it. If you don't include Turn 1 and its response in the messages list, Turn 2 has no context.  

**CCA architectural principle:** Clarity decisions compound across turns. A vague Turn 1 forces Claude to infer a task frame. Turn 2 can override that frame — but only if the history is accumulated. In production agentic applications, every turn adds to the context window and therefore to the token cost. This is why Turn 1 clarity matters architecturally, not just stylistically.

| Role | Purpose | Who provides it |
|---|---|---|
| `user` | Question, instruction, or content | The application / end user |
| `assistant` | Claude's response | Claude (from a prior API call) |

The list must **alternate roles** and must **start with `user`**. Sending two consecutive `user` messages without an `assistant` message between them will raise an API error.

In [ ]:
# Multi-turn demonstration: vague Turn 1 corrected by constrained Turn 2.
# This is a realistic scenario — a user sees a vague response and adds a constraint.
# Observe how Turn 2's clarity changes the response even though the underlying
# topic didn't change.

# ── TURN 1: Vague ─────────────────────────────────────────────────────────
# We build the messages list with a single user turn — the same as our ask() helper.
# We call client.messages.create() directly here so you can see the raw structure.

turn1_user_content = "Tell me about APIs."

# messages is a list — it starts with one dict and grows with each turn.
messages = [
    {"role": "user", "content": turn1_user_content}
    # role: 'user' means this is the human turn.
    # content: the text of the message.
]

# Send Turn 1 to the API.
response_turn1 = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=300,
    messages=messages         # One user message — single turn
)

# Extract the response text from the content block.
# response_turn1.content is a list; [0] gets the first (and only) TextBlock.
turn1_assistant_content = response_turn1.content[0].text

print("TURN 1 — Vague prompt:")
print(f'  User: "{turn1_user_content}"')
print(f"  Claude ({len(turn1_assistant_content.split())} words):")
print(turn1_assistant_content)
print()

# ── ACCUMULATE HISTORY ────────────────────────────────────────────────────
# Append Claude's response to the messages list as an 'assistant' turn.
# This is the history accumulation pattern — every response becomes the next
# assistant message so Claude has context in future turns.
messages.append({"role": "assistant", "content": turn1_assistant_content})
# messages now has 2 entries: [user, assistant]

# ── TURN 2: Corrective constraint ──────────────────────────────────────────
# The user sees the vague response and adds a direct, constrained follow-up.
# This demonstrates that clarity decisions can be recovered mid-conversation —
# but the recovery costs one extra turn (and therefore extra tokens).
turn2_user_content = (
    "That's too broad. Give me exactly one Python code example showing how to make "
    "a GET request to a public API using the requests library. "
    "Include the import statement. Maximum 6 lines of code. No prose before or after the code."
)

# Append the second user turn. messages now has 3 entries: [user, assistant, user]
messages.append({"role": "user", "content": turn2_user_content})

# Send Turn 2 with the full accumulated history.
# Claude sees both turns and responds in context.
response_turn2 = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=300,
    messages=messages         # Full history: [user, assistant, user]
)

turn2_assistant_content = response_turn2.content[0].text

print("TURN 2 — Corrective constraint:")
print(f'  User: "{turn2_user_content[:80]}..."')
print(f"  Claude ({len(turn2_assistant_content.split())} words):")
print(turn2_assistant_content)
print()

# ── ANALYSIS ──────────────────────────────────────────────────────────────
# Log the token cost of both turns so we can see the accumulation effect.
t1_tokens = response_turn1.usage.input_tokens + response_turn1.usage.output_tokens
t2_tokens = response_turn2.usage.input_tokens + response_turn2.usage.output_tokens

print("Token cost comparison:")
print(f"  Turn 1 total tokens : {t1_tokens}")
print(f"  Turn 2 total tokens : {t2_tokens}  ← includes full Turn 1 history in input")
print(f"  Overhead from Turn 1 history: ~{response_turn2.usage.input_tokens - response_turn1.usage.input_tokens} extra input tokens")
print()
print("CCA architectural insight:")
print("  A vague Turn 1 was recoverable — but recovery cost one extra API call.")
print("  In a production system with many users, that extra call multiplies at scale.")
print("  Getting Turn 1 right eliminates the recovery call entirely.")
print("  This is why first-line clarity is an architectural decision, not just good style.")

# Add to usage log for Section 9 analysis
usage_log.append({'label': 'multiturn_turn1', 'input_tokens': response_turn1.usage.input_tokens,
    'output_tokens': response_turn1.usage.output_tokens, 'stop_reason': response_turn1.stop_reason,
    'total_tokens': t1_tokens})
usage_log.append({'label': 'multiturn_turn2', 'input_tokens': response_turn2.usage.input_tokens,
    'output_tokens': response_turn2.usage.output_tokens, 'stop_reason': response_turn2.stop_reason,
    'total_tokens': t2_tokens})


---
## Section 6: Prompt property classification — vague, clear, direct, constrained
## CCA objective demonstrated: Operationalizing prompt properties as measurable boolean attributes

The lesson defines three prompt properties as **independent, stackable boolean flags**. A prompt can have any combination of them.

| Property | Definition | Signal in the prompt |
|---|---|---|
| **Clear** | Reduces ambiguity about subject and scope | Specific topic, defined audience, bounded domain |
| **Direct** | States the task explicitly with an imperative verb | Starts with or contains: Write, List, Explain, Classify, Summarize |
| **Constrained** | Specifies required output structure, fields, or limits | Mentions format, length, required sections, prohibited content |

These are **not synonyms** and they are **not a spectrum**. A prompt can be:
- Direct but not constrained: `"Explain recursion"` — imperative verb, no output spec
- Clear but not direct: `"About recursion in Python"` — scoped topic, no action verb
- Constrained but not direct: `"JSON with keys: answer, confidence"` — schema spec, no task stated
- All three: `"Explain recursion to a Python beginner in 3 bullet points, each under 20 words"`

**CCA exam heuristic:** `direct` controls *what Claude does*. `constrained` controls *what Claude produces*. `clear` controls *what Claude thinks it's talking about*. You need all three to fully specify a production prompt.

In [ ]:
# Operationalize the three properties as a classification function.
# In your question bank, every prompt version is tagged with these booleans.

def classify_prompt(prompt_text: str) -> dict:
    """
    Classify a prompt string against the three CCA prompt properties.
    Returns a dict of boolean flags, a property count, and a classification label.

    Statement-by-statement explanation follows inside the function body.
    """
    import re
    # re is Python's regular expression module.
    # We use it to scan prompt text for linguistic signals without calling the API.

    # ── DIRECT ──────────────────────────────────────────────────────────────
    # A direct prompt contains an imperative action verb that tells Claude exactly
    # what to do. We use re.search (not re.match) here deliberately:
    #   re.match  anchors to the START of the string — would miss "Please explain..."
    #   re.search scans the ENTIRE string — finds the verb wherever it appears.
    # This matters because prompts often begin with context before the verb.
    imperative_verbs = (
        r'\b(write|list|explain|classify|summarize|generate|describe|identify|'
        r'evaluate|compare|create|provide|analyze|define|outline|extract)\b'
    )
    # \b is a word boundary anchor — it matches the position between a word
    # character and a non-word character. This prevents 'explain' from matching
    # inside 'unexplained'. Without \b, 'unexplained' would incorrectly flag as direct.
    is_direct = bool(re.search(imperative_verbs, prompt_text.lower()))
    # bool() converts the Match object (truthy) or None (falsy) to True/False.

    # ── CLEAR ───────────────────────────────────────────────────────────────
    # Clarity is harder to detect mechanically than directness.
    # We use two heuristics combined with 'or' — either one is sufficient:
    #   1. Word count >= 6: very short prompts are almost always under-specified.
    #   2. Relational keywords: words like 'for', 'to', 'about' signal that the
    #      prompt is making a scope claim — specifying audience, domain, or context.
    # In production you would replace this with a classifier model.
    clarity_signals = r'\b(for|to|about|regarding|in the context of|assuming|given that|where)\b'
    is_clear = len(prompt_text.split()) >= 6 or bool(re.search(clarity_signals, prompt_text.lower()))
    # prompt_text.split() splits on whitespace and returns a list of words.
    # len() counts the list — a crude but fast word count.

    # ── CONSTRAINED ─────────────────────────────────────────────────────────
    # A constrained prompt specifies output shape: format, length, required fields,
    # or exclusion rules. We scan for structural vocabulary using re.search with \b
    # word boundaries (same reason as above — avoid partial word matches).
    constraint_signals = (
        r'\b(json|bullet|list|table|format|maximum|minimum|max|min|words|'
        r'sentences|sections|do not|avoid|never|only|exactly|must include|required)\b'
    )
    is_constrained = bool(re.search(constraint_signals, prompt_text.lower()))

    # ── CLASSIFICATION ──────────────────────────────────────────────────────
    # Count how many of the three properties are True.
    # sum([True, False, True]) == 2 because Python treats True as 1 and False as 0.
    score = sum([is_direct, is_clear, is_constrained])

    # Map the integer score to a human-readable label using list indexing.
    # Why a list rather than a dict lookup?
    #   score is guaranteed to be 0, 1, 2, or 3 — a perfect integer index.
    #   List indexing is O(1) and avoids a key lookup.
    #   A dict would require {0: 'vague', 1: 'minimal', ...} — more verbose for no gain.
    classification_labels = ["vague", "minimal", "partial", "fully_specified"]
    #                          [0]       [1]        [2]        [3]
    # score=0 → vague, score=1 → minimal, score=2 → partial, score=3 → fully_specified

    return {
        "direct": is_direct,
        "clear": is_clear,
        "constrained": is_constrained,
        "property_score": score,
        "classification": classification_labels[score]
    }


# Test the classifier on four prompts that span the property matrix
test_prompts = [
    "Tell me about APIs",
    "Explain what an API is",
    "Explain what a REST API is to a junior developer",
    "Explain REST APIs to a junior developer in exactly 3 bullet points, each under 20 words, without using the word 'endpoint'",
]

print(f"{'Prompt':<60} {'Direct':>7} {'Clear':>6} {'Const':>6} {'Score':>6} {'Class':<15}")
print("-" * 105)
for p in test_prompts:
    r = classify_prompt(p)
    display = (p[:57] + '...') if len(p) > 60 else p
    print(f"{display:<60} {str(r['direct']):>7} {str(r['clear']):>6} {str(r['constrained']):>6} {r['property_score']:>6} {r['classification']:<15}")


---
## Section 7: The improvement cycle — write → evaluate → improve → re-evaluate
## CCA objective demonstrated: Prompt iteration as a measurable, repeatable process

This is the core of the lesson. Three prompt versions are not just compared side-by-side — each version is scored with an objective rubric, and the score drives the decision to iterate.

> **Grader reliability note:** `score_response()` is a deterministic, keyword-based rubric — it is fast and reproducible, but it can over-score responses that contain the right words with wrong semantics. A response that says 'API' but misdefines it scores 1 for relevance. For production evals, complement deterministic checks with a Claude-as-judge semantic pass. This is exactly the rule + judge layering pattern you will build in the evaluation labs.

In [ ]:
# Define three prompt versions for the same underlying task.
# Task: explain REST APIs for a junior developer.

# v1: Vague — no imperative verb, no audience, no output structure
PROMPT_V1 = "Tell me about APIs."

# v2: Clear and direct — imperative verb + specific topic + audience
PROMPT_V2 = "Explain what a REST API is to a junior developer who knows Python but has never worked with web services."

# v3: Clear + direct + constrained — adds output structure and negative constraint
PROMPT_V3 = (
    "Explain what a REST API is to a junior developer who knows Python but has never worked with web services. "
    "Structure your response as exactly 3 sections: (1) What it is, (2) How it works, (3) A Python code example using the requests library. "
    "Each section must have a bold header. Do not use the phrase 'simply put' or other filler phrases. "
    "Total response must be under 250 words."
)

# Print property classifications for each version
for label, prompt in [("v1 (vague)", PROMPT_V1), ("v2 (clear+direct)", PROMPT_V2), ("v3 (fully specified)", PROMPT_V3)]:
    props = classify_prompt(prompt)
    print(f"{label}")
    print(f"  Direct: {props['direct']} | Clear: {props['clear']} | Constrained: {props['constrained']} | Classification: {props['classification']}")
    print()

In [ ]:
# Define an objective rubric for scoring responses.
# Each check returns 1 (pass) or 0 (fail). Maximum score: 5.
# CCA principle: quality must be measurable. Subjective review does not scale.

def score_response(response_text: str, prompt_version: str) -> dict:
    """
    Score a Claude response against a five-criterion rubric.
    Returns per-criterion scores, total, and a pass/fail boolean.

    Design rationale: each criterion is an independent binary check (0 or 1).
    Binary checks are deterministic and fast — they require no model calls.
    For richer semantic scoring, use a Claude-as-judge approach (see eval labs).

    Architect-level extension: replace the hard-coded criteria with a
    `rubric: dict` parameter so this function can score any task, not just
    the REST API explanation. That makes it the foundation of your eval pipeline.
    """
    import re

    # Normalize to lowercase for case-insensitive pattern matching.
    # We keep response_text (original case) for structure checks that depend on
    # Markdown syntax — ** and ## are case-sensitive by definition.
    text = response_text.lower()
    word_count = len(response_text.split())

    # ── CRITERION 1: Relevance ───────────────────────────────────────────────
    # re.search scans the full string (not just the start) for any REST/API term.
    # \b word boundaries prevent 'request' from matching inside 'requested'.
    # We use re.search here (not re.match) because the relevant term may appear
    # anywhere in the response — not necessarily at position 0.
    mentions_rest = int(bool(re.search(r'\b(rest|api|endpoint|http|request|response)\b', text)))
    # int(bool(...)) converts Match→True→1 or None→False→0.
    # This gives us a numeric score (0 or 1) rather than a boolean,
    # so we can sum all criteria at the end with a single sum() call.

    # ── CRITERION 2: Code example ────────────────────────────────────────────
    # Three signals are combined with 'or' — any one is sufficient evidence.
    #   '```' in response_text  : Markdown fenced code block (case-sensitive, original text)
    #   'requests' in text      : The Python requests library — common in API examples
    #   'import' in text        : Any import statement — evidence of runnable code
    has_code = int('```' in response_text or 'requests' in text or 'import' in text)

    # ── CRITERION 3: Structure ───────────────────────────────────────────────
    # Checks for any of: bold headers (**text**), ATX headers (##), numbered lists,
    # unordered lists with dash, unordered lists with asterisk.
    # We check response_text (original case) because Markdown is case-sensitive.
    # We use re.search because structural elements can appear anywhere in the response.
    has_structure = int(bool(re.search(r'(\*\*|##|\n\d+\.|\n-\s|\n\*\s)', response_text)))

    # ── CRITERION 4: Conciseness ─────────────────────────────────────────────
    # 300 words is a generous ceiling — the v3 prompt requests under 250.
    # We use a ceiling 20% above the prompt's stated limit to avoid false failures
    # from borderline word counts (e.g., a 252-word response that meets intent).
    is_concise = int(word_count <= 300)

    # ── CRITERION 5: No filler phrases ──────────────────────────────────────
    # \b word boundaries ensure we match whole phrases, not substrings.
    # For example, \bsimply put\b matches 'simply put' but not 'simply putting'.
    # The 'not bool(...)' inverts the logic: presence of filler = 0, absence = 1.
    filler_patterns = r'\b(simply put|in simple terms|as you can see|it is important to note|it is worth noting)\b'
    no_filler = int(not bool(re.search(filler_patterns, text)))

    # ── TOTAL ────────────────────────────────────────────────────────────────
    # Sum five binary integers. Maximum = 5, minimum = 0.
    total = mentions_rest + has_code + has_structure + is_concise + no_filler

    return {
        "version": prompt_version,
        "word_count": word_count,
        "mentions_rest": mentions_rest,
        "has_code": has_code,
        "has_structure": has_structure,
        "is_concise": is_concise,
        "no_filler": no_filler,
        "total_score": total,
        "max_score": 5,
        "pass": total >= 4   # Pass threshold: 4 of 5 criteria met
    }


print("Rubric defined. Pass threshold: 4/5 criteria.")
print("Criteria: REST relevance | Code example | Structure | Concise (≤300w) | No filler")


In [ ]:
# STEP 1: Generate and evaluate v1 (vague)
# Expected result: fails rubric — no structure, likely no code, may be off-topic

print("Generating v1 response...")
response_v1 = ask(PROMPT_V1, max_tokens=400, label="v1_vague")

score_v1 = score_response(response_v1, "v1_vague")

print(f"\n{'='*60}")
print("V1 RESPONSE:")
print(response_v1)
print(f"\n{'='*60}")
print("V1 RUBRIC SCORE:")
for k, v in score_v1.items():
    if k not in ('version',):
        icon = '✅' if v == 1 else ('❌' if v == 0 else '')
        print(f"  {k:<20}: {v} {icon}")

verdict = "PASS" if score_v1['pass'] else "FAIL — iterate"
print(f"\nVerdict: {verdict}")
if not score_v1['pass']:
    print("Action: improve the prompt and re-evaluate. Proceed to v2.")

In [ ]:
# STEP 2: Generate and evaluate v2 (clear + direct)
# Expected result: better than v1 but may lack structure and code

print("Generating v2 response...")
response_v2 = ask(PROMPT_V2, max_tokens=500, label="v2_clear_direct")

score_v2 = score_response(response_v2, "v2_clear_direct")

print(f"\n{'='*60}")
print("V2 RESPONSE:")
print(response_v2)
print(f"\n{'='*60}")
print("V2 RUBRIC SCORE:")
for k, v in score_v2.items():
    if k not in ('version',):
        icon = '✅' if v == 1 else ('❌' if v == 0 else '')
        print(f"  {k:<20}: {v} {icon}")

verdict = "PASS" if score_v2['pass'] else "FAIL — iterate"
print(f"\nVerdict: {verdict}")
if not score_v2['pass']:
    print("Action: add structural constraints. Proceed to v3.")

In [ ]:
# STEP 3: Generate and evaluate v3 (fully specified)
# Expected result: passes rubric — explicit structure, code, conciseness, no filler

print("Generating v3 response...")
response_v3 = ask(PROMPT_V3, max_tokens=600, label="v3_constrained")

score_v3 = score_response(response_v3, "v3_constrained")

print(f"\n{'='*60}")
print("V3 RESPONSE:")
print(response_v3)
print(f"\n{'='*60}")
print("V3 RUBRIC SCORE:")
for k, v in score_v3.items():
    if k not in ('version',):
        icon = '✅' if v == 1 else ('❌' if v == 0 else '')
        print(f"  {k:<20}: {v} {icon}")

verdict = "PASS" if score_v3['pass'] else "FAIL — further iteration needed"
print(f"\nVerdict: {verdict}")

In [ ]:
# STEP 4: Side-by-side comparison of all three versions
# This is the evidence that the improvement cycle produced measurable progress.

print("PROMPT IMPROVEMENT CYCLE — SUMMARY")
print("=" * 70)
print(f"{'Metric':<22} {'v1 Vague':>12} {'v2 Clear':>12} {'v3 Constrained':>15}")
print("-" * 70)

metrics = [
    ('Total score', 'total_score'),
    ('Word count', 'word_count'),
    ('REST relevance', 'mentions_rest'),
    ('Has code', 'has_code'),
    ('Has structure', 'has_structure'),
    ('Is concise', 'is_concise'),
    ('No filler', 'no_filler'),
]

for label, key in metrics:
    v1 = score_v1[key]
    v2 = score_v2[key]
    v3 = score_v3[key]
    print(f"{label:<22} {str(v1):>12} {str(v2):>12} {str(v3):>15}")

print("-" * 70)
print(f"{'Pass threshold (4/5)':<22} {'❌' if not score_v1['pass'] else '✅':>12} {'❌' if not score_v2['pass'] else '✅':>12} {'❌' if not score_v3['pass'] else '✅':>15}")
print()
print("Key insight: each version adds one or more prompt properties.")
print("Score improvement is directly correlated with property count — not luck.")

---
## Section 8: Failure mode analysis — what breaks when prompts are vague
## CCA objective demonstrated: Documenting production failure modes as architectural risk, not stylistic preference

Vague prompts are not just "less good" — they produce **specific, predictable failure modes** that manifest as production bugs. Understanding these failure modes is what separates prompt engineering from prompt guessing.

| Failure mode | What it looks like | Root cause | Fix |
|---|---|---|---|
| **Hallucinated structure** | Claude invents sections not requested — adds "Conclusion", "Further reading", "Disclaimer" | No output schema specified | Add constrained section list |
| **Variable format** | Sometimes prose, sometimes bullets, sometimes numbered — inconsistent across API calls | No format instruction | Specify format explicitly in every call |
| **Wrong audience register** | Too technical for beginners, too basic for experts | No audience specification | Add explicit audience description |
| **Length explosion** | 800-word response when 100 words was sufficient | No length constraint | Add word or sentence count |
| **Scope creep** | Asked about REST, also explains SOAP, GraphQL, gRPC, WebSockets | Task frame is open-ended question, not bounded instruction | Use imperative verb + explicit scope |
| **Silent assumption** | Assumes weight loss as the goal when user said "get healthier" | Underspecified goal, model infers most plausible interpretation | Add explicit goal or require assumption disclosure |
| **Truncation from over-constraining** | Response cuts off mid-sentence because too many constraints inflated token count | Too many constraints + insufficient max_tokens | Triage constraints — keep non-negotiables, replace style rules with examples |

In [ ]:
# Demonstrate two failure modes live: variable format and scope creep
# Run the vague prompt multiple times and observe output variance.

VAGUE_PROMPT = "Tell me about machine learning."

print("Demonstrating output variance from a vague prompt.")
print("Same prompt, same model, temperature=1.0 (default) — watch what varies.")
print()

results = []
for i in range(3):
    r = ask(VAGUE_PROMPT, max_tokens=300, temperature=1.0, label=f"failure_demo_{i}")
    word_count = len(r.split())
    has_bullets = '- ' in r or '* ' in r
    has_numbered = bool(__import__('re').search(r'\n\d+\.', r))
    results.append({'run': i+1, 'words': word_count, 'bullets': has_bullets, 'numbered': has_numbered, 'text': r})
    print(f"--- Run {i+1} ---")
    print(f"Word count: {word_count} | Bullets: {has_bullets} | Numbered: {has_numbered}")
    print(r[:200] + ('...' if len(r) > 200 else ''))
    print()

print("=" * 60)
print("Variance summary:")
words = [r['words'] for r in results]
print(f"  Word count range: {min(words)} – {max(words)} (delta: {max(words)-min(words)})")
print(f"  Format consistency: {'consistent' if len(set(r['bullets'] for r in results)) == 1 else 'INCONSISTENT — different formats across runs'}")
print()
print("This variance is a production bug. A downstream parser expecting bullets")
print("will fail on the runs that produce prose.")

---
## Section 9: Token usage analysis
## CCA objective demonstrated: Measuring API cost as an output of prompt design decisions

In [ ]:
# Analyze token usage across all calls in this session.
# CCA principle: prompt design decisions have measurable cost consequences.
# Constrained prompts can reduce output token variance — reducing both cost and latency.

print("TOKEN USAGE LOG — This session")
print("=" * 75)
print(f"{'Label':<30} {'Input':>8} {'Output':>8} {'Total':>8} {'Stop reason':<15}")
print("-" * 75)

for entry in usage_log:
    print(f"{entry['label']:<30} {entry['input_tokens']:>8} {entry['output_tokens']:>8} {entry['total_tokens']:>8} {entry['stop_reason']:<15}")

print("-" * 75)

if usage_log:
    total_input = sum(e['input_tokens'] for e in usage_log)
    total_output = sum(e['output_tokens'] for e in usage_log)
    print(f"{'TOTALS':<30} {total_input:>8} {total_output:>8} {total_input+total_output:>8}")
    print()

    # Extract v1/v2/v3 entries and compare output tokens
    version_entries = {e['label']: e for e in usage_log if e['label'] in ('v1_vague', 'v2_clear_direct', 'v3_constrained')}
    if len(version_entries) == 3:
        print("Output token comparison for prompt versions:")
        for label in ('v1_vague', 'v2_clear_direct', 'v3_constrained'):
            e = version_entries[label]
            print(f"  {label:<25}: {e['output_tokens']} output tokens")
        print()
        print("Observation: constrained prompts often reduce output token variance.")
        print("Lower variance = more predictable latency and cost at scale.")

    # Flag any truncated responses
    truncated = [e for e in usage_log if e['stop_reason'] == 'max_tokens']
    if truncated:
        print(f"\n⚠️  {len(truncated)} response(s) were truncated (stop_reason=max_tokens):")
        for e in truncated:
            print(f"   {e['label']} — increase max_tokens for this call")

---
## Section 10: Practice drills — apply the improvement cycle to new prompts
## CCA objective demonstrated: Independent application of prompt property classification and iteration

In [ ]:
# DRILL 1: Classify and improve this prompt
# Starting prompt: "Explain cloud computing"
# Your task: produce a v3 that scores 5/5 on the rubric

DRILL_1_START = "Explain cloud computing"

# Step 1: Classify the starting prompt
drill1_props = classify_prompt(DRILL_1_START)
print("Starting prompt classification:")
print(f"  {DRILL_1_START}")
print(f"  Direct: {drill1_props['direct']} | Clear: {drill1_props['clear']} | Constrained: {drill1_props['constrained']}")
print(f"  Classification: {drill1_props['classification']}")
print()

# Step 2: Write your improved v3 prompt here
# Replace the placeholder with a fully specified version
DRILL_1_V3 = (
    "Explain the three main cloud computing service models (IaaS, PaaS, SaaS) "
    "to a non-technical product manager who needs to choose between them for a new application. "
    "Use a table with columns: Model, Who manages what, Best for. "
    "Follow the table with a one-sentence recommendation. Do not use acronyms without defining them. "
    "Maximum 150 words."
)

# Step 3: Generate and score
response_drill1 = ask(DRILL_1_V3, max_tokens=400, label="drill_1_v3")
score_drill1 = score_response(response_drill1, "drill_1_v3")

print("DRILL 1 RESPONSE:")
print(response_drill1)
print()
print(f"Score: {score_drill1['total_score']}/5 | Pass: {score_drill1['pass']}")
print("If score < 5: identify which criterion failed and revise the prompt.")

In [ ]:
# DRILL 2: System prompt design
# Design a system prompt that makes the same user question produce
# appropriate responses for two different audiences.

USER_QUESTION = "What is a database?"

# Your task: fill in the two system prompts below
SYSTEM_ANALYST = (
    "You are a senior data architect writing documentation for business analysts "
    "who understand data concepts but do not write SQL. "
    "Be precise. Use business terminology. Maximum 4 sentences."
)

SYSTEM_EXECUTIVE = (
    "You are explaining technology to a C-suite executive who wants business implications only. "
    "Never use technical terms. Use a business analogy. Maximum 3 sentences."
)

print(f"User question: '{USER_QUESTION}'")
print()

r_analyst = ask(USER_QUESTION, system=SYSTEM_ANALYST, max_tokens=200, label="drill_2_analyst")
print("SYSTEM: Data analyst audience")
print(r_analyst)
print()

r_executive = ask(USER_QUESTION, system=SYSTEM_EXECUTIVE, max_tokens=200, label="drill_2_executive")
print("SYSTEM: Executive audience")
print(r_executive)
print()

print("CCA principle: the system parameter is not a tone dial — it's an audience and constraint framework.")
print("In production, inject the right system context per user role at request time.")

In [ ]:
# DRILL 3: Negative constraint precision
# Practice writing negative constraints that actually close the loop.
# Each attempt is scored on whether the prohibited behavior is absent.

import re

# Scenario: legal tech product — summaries must not contain legal advice.
# This is a real production constraint: many legal tech companies cannot let
# their AI products give advice that constitutes the unauthorized practice of law.
CONTRACT_TEXT = """
This agreement grants Party A a non-exclusive, revocable license to use the software.
Party B retains all intellectual property rights. Section 4.2 allows termination with 30 days notice.
Liability is capped at the amount paid in the preceding 12 months.
"""

# v1: Positive instruction only — no prohibition on advisory language.
# Claude's helpful defaults include flagging important clauses — that's a problem here.
LEGAL_PROMPT_V1 = f"Summarize this contract clearly:\n{CONTRACT_TEXT}"

# v2: Explicit negative constraint — names the prohibited output class.
# Note the three-layer prohibition: category ('legal advice'), type ('recommendations'),
# and linguistic signals ('you should', 'you may want to', 'consider').
# Redundancy is intentional — Claude may interpret 'legal advice' narrowly.
LEGAL_PROMPT_V2 = (
    f"Summarize the key terms of this contract in 4 bullet points.\n{CONTRACT_TEXT}\n\n"
    "Do not include legal advice, recommendations, or any language suggesting the reader take action. "
    "Report facts only. Do not use the phrases 'you should', 'you may want to', or 'consider'."
)


def check_no_legal_advice(text: str) -> dict:
    """
    Scan a response for advisory language patterns.
    Returns whether any were found, and which ones.

    Statement-by-statement explanation:
    """
    # advisory_patterns is a list of raw regex strings.
    # Each pattern targets one linguistic form of advisory language.
    # We use \b word boundaries throughout — same reason as in classify_prompt:
    # prevents 'consider' from matching inside 'reconsider' or 'considerable'.
    # Without \b, 'reconsider your options' would incorrectly trigger the 'consider' pattern.
    advisory_patterns = [
        r'\byou should\b',        # Direct recommendation
        r'\byou may want\b',      # Softer recommendation (often used to avoid sounding directive)
        r'\bconsider\b',          # Implicit recommendation
        r'\bwe recommend\b',      # Explicit recommendation
        r'\badvise\b',            # Legal advisory language
        r'\bconsult\b',           # 'Consult an attorney' — the classic legal advice signal
        r'\bit is advisable\b',   # Formal advisory phrasing
        r'\bensure that you\b',   # Action directive disguised as instruction
    ]

    # List comprehension: iterate over patterns, keep those that match.
    # re.search(pattern, text.lower()) returns a Match object if found, None if not.
    # The list comprehension collects only the patterns where re.search is truthy.
    # We search text.lower() for case-insensitive matching — advisory language
    # appears regardless of capitalization.
    found = [p for p in advisory_patterns if re.search(p, text.lower())]

    # bool(found) is True if the list is non-empty (at least one pattern matched).
    return {'advisory_language_found': bool(found), 'patterns': found}


print("Testing negative constraint effectiveness...")
print()

r_legal_v1 = ask(LEGAL_PROMPT_V1, max_tokens=300, label="legal_v1")
check_v1 = check_no_legal_advice(r_legal_v1)
print("V1 (no negative constraint):")
print(r_legal_v1)
print(f"Advisory language detected: {check_v1['advisory_language_found']} | Patterns: {check_v1['patterns']}")
print()

r_legal_v2 = ask(LEGAL_PROMPT_V2, max_tokens=300, label="legal_v2")
check_v2 = check_no_legal_advice(r_legal_v2)
print("V2 (explicit negative constraint):")
print(r_legal_v2)
print(f"Advisory language detected: {check_v2['advisory_language_found']} | Patterns: {check_v2['patterns']}")
print()
print("CCA principle: 'Be objective' does not prohibit advisory language — Claude can be objective")
print("and still give advice. Only explicit, named prohibitions close the loop.")
print("The \\b word boundary in each pattern is what makes this detection reliable —")
print("without it, 'reconsider' would trigger the 'consider' check incorrectly.")


---
## Section 11: CCA takeaways and exam-ready mental models
## CCA objective demonstrated: Synthesis of lesson concepts into transferable architectural heuristics

### 5 CCA exam-ready principles from this lab

**1. Direct, clear, and constrained are independent boolean properties — not a spectrum.**  
A prompt can be any combination of them. Know the 2×2×2 matrix. The CCA exam tests whether you can classify and prescribe each property independently.

**2. The system parameter is an architectural decision, not a convenience.**  
Role, audience, persistent format constraints, and negative constraints belong in `system`. Task-specific inputs belong in the user turn. In multi-tenant systems, the system prompt is a parameterized template filled at request time — not a fixed string.

**3. Prompt improvement is a measurable, repeatable cycle — not subjective editing.**  
Write → evaluate against an objective rubric → identify which criterion fails → add the missing property → re-evaluate. The rubric is what separates engineering from guessing.

**4. Negative constraints are first-class prompt citizens.**  
For any domain with regulatory or liability exposure, enumerate what Claude must not produce. The test: could Claude follow all your positive instructions and still generate the unwanted output? If yes, add an explicit prohibition.

**5. Vague prompts produce specific, predictable failure modes — not just "worse" outputs.**  
Variable format, scope creep, wrong audience register, silent assumptions, and length explosion are production bugs — they break downstream systems, not just aesthetics. The fix is always upstream (prompt design), not downstream (post-processing).

---

### Lab completion checklist

| Section | CCA competency | Completed |
|---|---|---|
| Section 3 | API helper with token logging | ☐ |
| Section 4 | Error handling — required parameters | ☐ |
| Section 5 | System parameter usage and audience framing | ☐ |
| Section 6 | Prompt property classification | ☐ |
| Section 7 | Write → evaluate → improve → re-evaluate cycle | ☐ |
| Section 8 | Failure mode documentation | ☐ |
| Section 9 | Token usage analysis | ☐ |
| Section 10 | Independent application — three drills | ☐ |

---
## Next steps

- Proceed to the CCA question set for this lesson — questions are keyed to the architectural decisions exercised in Sections 6, 7, 8, and 10.
- Review the token usage log (Section 9) and note which prompt version produced the most predictable output token count.
- Architect-level extension: modify `score_response()` to accept a custom rubric dict so it can evaluate any task, not just the REST API explanation. This is the foundation of the evaluation pipeline in later labs.